In [ ]:
import os
import torch
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from torchvision.io import decode_image
from torchvision.ops import nms
from torchvision import tv_tensors
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.ops import generalized_box_iou_loss
from torch import nn
import numpy as np
import wandb
from copy import deepcopy
from dotenv import load_dotenv

In [ ]:
torch.manual_seed(42)

In [ ]:
load_dotenv() 

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [4]:
weights = ResNet50_Weights.DEFAULT # weights = IMAGENET1K_V2
preprocess = weights.transforms()

In [5]:
# double input size tensor to have more spacial information and have less collision do to a greater number of cells
preprocess.resize_size[0] *= 2
preprocess.crop_size[0] *= 2
image_size = preprocess.crop_size[0]
print(preprocess)

ImageClassification(
    crop_size=[448]
    resize_size=[464]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [6]:
model = resnet50(weights=weights) 
for param in model.parameters():
    param.requires_grad = False # freeze the parameters of the model to only train the new layers for object detection first

## Build Dataloader loading the Custom Dataset

In [7]:
# Define the custom dataset and how to load the images and annotations => images and bounding boxes have to transformed equally

class CustomImageDataset(Dataset):
    def __init__(self, annotations_dir, img_dir, transform=None, img_transform=None):
        self.annotations_dir = annotations_dir
        self.annot_file_names = os.listdir(self.annotations_dir)
        self.img_dir = img_dir
        self.transform = transform  # v2 transformations which will be applied to image + label
        self.img_transform = img_transform # normal transformations (which will be also applied to validation + test) like resize, crop and normalization

    def __len__(self):
        return len(self.annot_file_names)

    def __getitem__(self, idx):
        file_name = self.annot_file_names[idx]
        # build the path to the annotation.txt and read the content
        annotation_path = os.path.join(self.annotations_dir, file_name)
        with open(annotation_path, "r") as f:
            try:
                objects = f.read().strip().split("\n")
                yolo_boxes = [[float(x.strip()) for x in obj.strip().split(" ")] for obj in objects]  # transform the string to a readable float label array
                # annotations as lines in txt-file with no header, each line containes 5 values seperated by a space: (class_id, x_center, y_center, width, height)
            except Exception as e:
                # no object/label for this image => file is empty, so use an empty label-list as the GT-annotation
                yolo_boxes = []
        # build the related path to the image and load it as a tensor
        img_path = os.path.join(self.img_dir, file_name).removesuffix("txt") + "jpg"
        image = decode_image(img_path)
        current_img_size = image.shape[-1] # image is a tensor with shape [C, H, W]
        # apply transformations if necessary
        if self.transform:
            if len(yolo_boxes) > 0: # only do the bounding box transformations if there are bounding boxes in the image
                raw_boxes = np.array([[cx, cy, w, h] for _, cx, cy, w, h in yolo_boxes]) 
                cxcywh_boxes = raw_boxes * current_img_size # cxcywh boxes is noted with absolute pixels, yolo with relative from [0, 1], so multiply by image size
                # => boxes are relative to the original image size, so we multiply by the current image size (instead of the target image size) to get the current absolute pixel values for the bounding box
                bboxes = tv_tensors.BoundingBoxes(cxcywh_boxes, format="CXCYWH", canvas_size=(current_img_size, current_img_size)) # canvas_size is the size of the original image
                image, transformed_bboxes = self.transform(image, bboxes) # transform both the image and the bounding boxes (only with v2 transformations which also affect the bounding boxes)
                relative_bboxes = transformed_bboxes / image.shape[-1] # scale back to relative bounding boxes ([0, 1]) for the yolo format => now relative to the new image size

                # to convert back to the yolo label format, the class labels have to be added again
                classes = torch.tensor([lbl[0] for lbl in yolo_boxes]).unsqueeze(1) # unsqueeze(1) to get a shape of [N, 1] => bboxes are shape [N, 4] so we can concatenate them to get a shape of [N, 5] for the yolo_boxes
                yolo_boxes = torch.cat((classes, relative_bboxes), dim=1) # dim=1 to concatenate along the columns 
            else:
                image = self.transform(image) # if there are no bounding boxes, just transform the image
        if self.img_transform:
            # preprocess transformations like normalize, which will not affect the label
            image = self.img_transform(image)
        return image, yolo_boxes

In [ ]:
# Prepare Datasets

transform = v2.Compose([
    v2.RandomResizedCrop(image_size, (0.6, 1)),
    v2.RandomPerspective(0.75, 0.2),
    v2.ColorJitter(0.3, 0.2, 0.2, 0.1),
    v2.RandomGrayscale(0.1)
])

dataset_path = "../kaggle/Traffic_Sign/car"

train_dataset =  CustomImageDataset(
    annotations_dir=f"{dataset_path}/train/labels",
    img_dir=f"{dataset_path}/train/images",
    transform=transform,
    img_transform=preprocess
)

val_dataset =  CustomImageDataset(
    annotations_dir=f"{dataset_path}/valid/labels",
    img_dir=f"{dataset_path}/valid/images",
    img_transform=preprocess
)

In [10]:
# load class names from yaml file

import yaml

with open(f"{dataset_path}/data.yaml", "r") as f:
    class_names = yaml.safe_load(f)["names"]

print("Num classes:", len(class_names))
print("Classes:", class_names)

Num classes: 15
Classes: ['Green Light', 'Red Light', 'Speed Limit 10', 'Speed Limit 100', 'Speed Limit 110', 'Speed Limit 120', 'Speed Limit 20', 'Speed Limit 30', 'Speed Limit 40', 'Speed Limit 50', 'Speed Limit 60', 'Speed Limit 70', 'Speed Limit 80', 'Speed Limit 90', 'Stop']


In [ ]:
# Prepare Dataloader and define collate function to handle the different amounts of bounding boxes in each image

batch_size = 64
num_workers = 0 # multiple workers will not work in a ipynb on macOS

def collate_fn(batch):
    return list(zip(*batch))
    # rearrange the batch into two tuples, each with len()=64: [(image_1, image_2, ...), (target_1, target_2, ...)]

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)

In [12]:
def get_box_coords(box, image_size):
    class_id, x_c, y_c, w, h = box
    width = w * image_size
    height = h * image_size
    x1 = x_c * image_size - width / 2
    y1 = y_c * image_size - height / 2
    return (x1, y1, width, height)

## Changing Architecture to Object Detection instead of Classification

In [ ]:
# building the model so the output is [64, 20, 14, 14] as we have 15 class probabilities + 4 coordinates + 1 objectness score with B = 1

class ObjectDetectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=2048, out_channels=2048, kernel_size=3, padding=1, bias=False)
        self.batchnorm1 = nn.BatchNorm2d(2048)
        self.conv2 = nn.Conv2d(in_channels=2048, out_channels=2048, kernel_size=3, padding=1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(2048)
        self.outputmapping = nn.Conv2d(in_channels=2048, out_channels=20, kernel_size=1, bias=True)


    def forward(self, x):
        # extra Layer 1
        x = F.relu(self.batchnorm1(self.conv1(x)))
        # extra Layer 2
        x = F.relu(self.batchnorm2(self.conv2(x)))
        # mapping on desired output tensor shape
        x = self.outputmapping(x)
        return x
    
model = nn.Sequential(*list(model.children())[:-2]) # remove last two layers (avg_pool and fc) from the model 

model.add_module("Object Detection Head", ObjectDetectionHead())
model.to(device)

## Defining Loss Function and Calculation
- Objectness Loss: Sigmoid + BCE => an image can have multiple objects
- Classification Loss: Softmax + CE (only on cells with objectness = 1)
- Localization Loss: Sigmoid + GIoU + (MSE for w and h) => only GIoU is leading to an unstable training due to quick convergence into close-to-zero-gradient areas (also only on cells with objectness = 1)

In [ ]:
# defining weights for the loss calculation
no_obj_w = 0.2 
loc_w = 4
obj_w = 12
class_w = 1
loc_l2_factor = 1

In [ ]:
# tensor shape: [batch_size, 20, 14, 14]

# feature map assignment
# (0)           objectness
# (1)           bounding box - x (relative to cell border)
# (2)           bounding box - y (relative to cell border)
# (3)           bounding box - w (relative to image size)
# (4)           bounding box - h (relative to image size)
# (5) - (19)    class prob per cell 


def get_target_maps(map_size, batch_size, y, device=None):
    # to get the correct 2D Feature map for the objectness "map" for each batch item, get the x_center and y_center (relative to image size) of all objects, multiple by image size (14) and store in a tensor
    obj_map = torch.zeros(batch_size, map_size, map_size).to(device)
    class_maps = torch.ones(batch_size, map_size, map_size, dtype=torch.long).to(device) * -1
    loc_maps = torch.zeros(batch_size, 4, map_size, map_size).to(device)

    for b_idx, b_item in enumerate(y):
        for object in b_item:
            (c, x_cent, y_cent, w, h) = object
            x_coord, y_coord = min(13, int(x_cent * map_size)), min(13, int(y_cent * map_size)) # min(13, ...) because when x_cent or y_cent is 1.0 an OutOfBoundsError would occur (index = 14)
            x_offset, y_offset = (x_cent * map_size - x_coord), (y_cent * map_size - y_coord)

            obj_map[b_idx, y_coord, x_coord] = 1
            class_maps[b_idx, y_coord, x_coord] = int(c)
            loc_maps[b_idx, :, y_coord, x_coord] = torch.tensor([x_offset, y_offset, w, h]) # inject the id tensor along the second axis by getting all element of that pixel for alle 4 feature maps
        
    return obj_map, class_maps, loc_maps


def yolo_to_bbox(yolo_box, img_size, map_size):
    # 64, 4, 14, 14
    cell_width = img_size / map_size

    # yolo_box[:, 0:2] is now only the x-offset and the y-offset feature map, but for GIoU we need the absolute coordinates of x1 and y1
    x_scale_tensor = torch.arange(0, map_size).expand(map_size, -1).to(device) # arange creates vector [0, ..., map_size-1], expand duplicates that to reach shape (map_size x map_size)
    y_scale_tensor = torch.arange(0, map_size).view((-1, 1)).expand(-1, map_size).to(device) # here we need to change the row vector to a "column vector", then expand "to the right"
    # calculate absolute x-coordinates => every column needs to be multiplied by 32 times the column index
    abs_x = cell_width * x_scale_tensor + yolo_box[:, 0] * cell_width 
    # calculate absolute y-coordinates => same as x-coordinate calculation but with 32 times the row index
    abs_y = cell_width * y_scale_tensor + yolo_box[:, 1] * cell_width
    
    # also convert relative height/width into absolute height/width
    abs_w = yolo_box[:, 2] * img_size
    abs_h = yolo_box[:, 3] * img_size

    x1 = abs_x - abs_w / 2
    y1 = abs_y - abs_h / 2
    x2 = abs_x + abs_w / 2
    y2 = abs_y + abs_h / 2

    bbox_tensor = torch.stack((x1, y1, x2, y2), dim=3)
    return bbox_tensor


def objectness_loss(pred, target_obj):
    # using Sigmoid + BCE for the objectness score as there can be mulitple objects in an image (higher weight on objectness = 1 because there are more pixels with no objects than with objects)
    pred_obj = pred[:, 0] 
    loss_obj = F.binary_cross_entropy_with_logits(input=pred_obj, target=target_obj.float(), reduction="none") # this function automatically applies sigmoid function to the logits
    weight_map = torch.where(target_obj == 1, torch.ones_like(loss_obj), torch.ones_like(loss_obj) * no_obj_w) 
    loss_obj *= weight_map # multiply by no_obj_w, where no object should be detected - inplace with masked_fill_
    # as there are way more pixels with no objects, weight no object pixels less 
    # with standard reduction = "mean" the mean is taken over every pixel over every batch so Sum_of_BCE / (64 * 14 * 14) 
    return loss_obj.mean()


def localization_loss(pred, target_loc, img_size, map_size, target_obj):
    # using GIoU + L2 distance penalty for width and height => GIoU to normalize for different sizes of objects
    # by only using GIoU tho, the model can't confidently localize small objects, which is adressed by the L2 distance penalty => encourages for smaller boxes, helping the model get out of the small gradients plateau at the beginning of training 
    yolo_pred = torch.sigmoid(pred[:, 1:5])
    bbox_pred = yolo_to_bbox(yolo_pred, img_size=img_size, map_size=map_size)
    bbox_target = yolo_to_bbox(target_loc, img_size=img_size, map_size=map_size)
    # prediction and target localization tensors have shape [64, 14, 14, 4], but for GIoU loss, shape [64, 14, 14, 4] is needed

    all_giou_loss = generalized_box_iou_loss(bbox_pred, bbox_target)
    l2_w = loc_l2_factor * (yolo_pred[:, 2].sqrt() - target_loc[:, 2].sqrt()) ** 2
    l2_h = loc_l2_factor * (yolo_pred[:, 3].sqrt() - target_loc[:, 3].sqrt()) ** 2
    # the distance penalty (like YOLOv1 does it) is necessary, because otherwise the model does not correctly localize small images
    # when only using the GIoU-Loss, the gradient is really small with a small IoU and then oszillates when breaking out of that plateau (doesn't even get out of the plateau with normal batch sizes, like e.g. 64)
    # boxes stay large with the GT box in that pred box (GIoU equals roughly IoU, so GIoU-Loss around 1) => with the distnace penalty we push the model out of that pleateu by encouriging smaller width and height
    giou_l2_loss = all_giou_loss + l2_w + l2_h
    mean_relevant_giou_loss = torch.masked_select(giou_l2_loss, target_obj > 0).mean()

    return mean_relevant_giou_loss


def classification_loss(pred, target_class):
    # Softmax + Cross Entropy for the classification loss (only one object per cell can be detected, so only one class is possible) => loss is only calculated on cells with objectness = 1
    class_logits = pred[:, 5:]
    loss_class = F.cross_entropy(class_logits, target_class, reduction="mean", ignore_index=-1) # ignore all indices with no class assignnment (objectness = 0) => class = -1
    # Cross Entropy loss automaticall calculates the softmax over dim =1 (softmax for each pixel between all feature maps) and compares with the correct class index with target_value = 1 for that class
    # with standard reduction = "mean" the mean is taken over the loss for every pixel over every batch so Sum_of_BCE / (64 * 14 * 14) => the 25 dimensions from the classes are reduced to one dimension (CE loss)
    return loss_class



def calc_detection_loss(pred, y, img_size=image_size, device=device):
    # calculates all three losses and weights them to account for the different orders of magnitude and importances
    map_size = pred.shape[-1]

    target_obj, target_class, target_loc = get_target_maps(map_size=map_size, batch_size=pred.shape[0], y=y, device=device)
    
    loss_obj = objectness_loss(pred, target_obj)
    loss_class = classification_loss(pred, target_class)
    loss_loc = localization_loss(pred, target_loc, img_size=img_size, map_size=map_size, target_obj=target_obj)


    loss = obj_w * loss_obj + class_w * loss_class + loc_w * loss_loc
    return loss, obj_w * loss_obj, class_w * loss_class, loc_w * loss_loc


### Prepare for training and prediction visualizaion

In [ ]:
def train_loop(model, dataloader, optimizer):
    model.train()

    batch_losses = []
    all_batch_losses = []

    for batch in dataloader:
        X, y = torch.stack(batch[0]).to(device), batch[1]

        pred = model(X)
        loss, loss_obj, loss_class, loss_loc = calc_detection_loss(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        batch_losses.append(loss.item())
        all_batch_losses.append((loss.item(), loss_obj.item(), loss_class.item(), loss_loc.item()))

    return all_batch_losses

        
def test_loop(model, dataloader):
    model.eval() 

    batch_losses = []
    all_batch_losses = []

    with torch.no_grad():
        for batch in dataloader:
            X, y = torch.stack(batch[0]).to(device), batch[1]

            pred = model(X)
            loss, loss_obj, loss_class, loss_loc = calc_detection_loss(pred, y)
            batch_losses.append(loss.item())

            all_batch_losses.append((loss.item(), loss_obj.item(), loss_class.item(), loss_loc.item()))

    return all_batch_losses



In [ ]:
def get_coords_from_pred(pred, threshold, image_size):
    class_maps = torch.softmax(pred[:, 5:], dim=1)
    max_probs, class_ids = torch.max(class_maps, dim=1)

    obj_map = torch.sigmoid(pred[:, 0])
    cell_confs = obj_map * max_probs # to get the total confidence for each cell, multiply the cell confidence with the class probability
    obj_mask = cell_confs > threshold
    obj_indices = torch.nonzero(obj_mask, as_tuple=True)
    # obj_indices list of length 3, each with 64 elements
    # first element are the batch indices, second element the y-coordinates (row) and third element the x-coordinates (column) of cells with objectness > 0.2

    obj_classes = class_ids[obj_indices] # because obj_indices is a tuple (batch_idx_tensor, y_coord_tensor, x_coord_tensor) we can index like class_ids[batch_idx_tensor, y_coord_tensor, x_coord_tensor]
    obj_conf = cell_confs[obj_indices] 

    loc_maps = torch.sigmoid(pred[:, 1:5])
    bbox_preds = yolo_to_bbox(loc_maps, img_size=image_size, map_size=pred.shape[-1])
    obj_bboxes = bbox_preds[obj_indices]

    return torch.stack(obj_indices, dim=1), obj_classes, obj_conf, obj_bboxes


In [ ]:
def get_bbox_preds(batch_size, pred, threshold=0.25, image_size=image_size, nms_treshold=0.5):
    obj_indices, classes, confidences, bboxes = get_coords_from_pred(pred, threshold=threshold, image_size=image_size)

    nms_classes = []
    nms_bboxes = []
    nms_confidences = []

    for i in range(batch_size):
        # draw the boxes around the correct objects
        pred_mask = obj_indices[:, 0] == i
        obj_conf = confidences[pred_mask] # confidence scores of all predictions above the threshold for the current image
        obj_bboxes = bboxes[pred_mask]  # bbox predictions for object predictions above the threshold for the current image

        nms_indices = nms(obj_bboxes, obj_conf, iou_threshold=nms_treshold) # return the indices of the bbox-tensor (pred_loc) that are kept after nms
        nms_preds_indices = torch.nonzero(pred_mask)[nms_indices] # because the indices are relative to the pred_loc tensor, we calculate the absolute indices of the predictions (to index the ouput of get_coords_from_pred())

        nms_pred_classes = classes[nms_preds_indices] # all relevant classes after nms
        nms_pred_bboxes = obj_bboxes[nms_indices] # filter the bbox predictions to only keep the ones that are kept after nms
        nms_pred_conf = obj_conf[nms_indices]

        nms_classes.append(nms_pred_classes)
        nms_bboxes.append(nms_pred_bboxes)
        nms_confidences.append(nms_pred_conf)

    return nms_classes, nms_bboxes, nms_confidences

In [ ]:
def visualize_bbox_preds(X, pred_classes, pred_bboxes, confidences, y=None, image_size=image_size):
    fig, axs = plt.subplots(4, 4, figsize=(15, 12))
    axs = axs.flatten()

    mean = torch.tensor(preprocess.mean).view(3, 1, 1) # because we have a tensor with shape [3, 244, 244], to process each channel seperately, we have to create a tensor with shape [3, 1, 1]
    std  = torch.tensor(preprocess.std).view(3, 1, 1)

    # visualize the first 16 images in the batch
    for i in range(16):
        # show the image with the classified label
        # X[i] is the tensor of the normalized image (with mean and std defined above) => with those parameters, we convert back to the normal image
        normal_image = X[i] * std + mean # revert the normalization process by multiplying with the standard deviation and adding the mean
        
        ax = axs[i]
        ax.imshow(normal_image.permute(1, 2, 0))
        ax.axis("off")
        
        
        # iterate over every prediction for the current image, get the coordinates and draw them into the current image
        for (cls, loc, conf) in zip(pred_classes[i], pred_bboxes[i], confidences[i]):
            x1, y1, x2, y2 = loc.cpu().detach().numpy()
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=2))
            pred_class_idx = cls.item()
            ax.annotate(f"{class_names[pred_class_idx]} - {conf.item() * 100:.2f}%", 
                        xy=(x1, y1 - image_size * 0.02), 
                        color='black',
                        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'), 
                        fontsize=8)
        # correct prediction in green (if available)
        if y:
            x1_gt, y1_gt, w_gt, h_gt = get_box_coords(y[i][0], image_size)
            ax.add_patch(plt.Rectangle((x1_gt, y1_gt), w_gt, h_gt, fill=False, edgecolor='green', linewidth=2))

        # => torchvision.util.draw_bounding_boxes() can also draw bounding boxes in a given image

    plt.tight_layout()
    plt.show()


## Training model head on whole Dataset

In [ ]:
def train_model(model, train_dataloader, val_dataloader, optimizer, wandb_project_name, wandb_config, epochs):
    train_losses = []
    val_losses = []
    best_model_dict = None
    best_val_loss = float("inf")

    labels = ["Total_Loss", "Objectness_Loss", "Classification_Loss", "Localization_Loss"]

    with wandb.init(project=wandb_project_name, config=wandb_config) as run:
        run.watch(model)

        for epoch in range(epochs):
            print(f"-------------------------------\nEpoch {epoch+1}")

            train_batch_losses = train_loop(model, train_dataloader, optimizer)
            val_batch_losses = test_loop(model, val_dataloader)


            mean_train_losses = [np.array(l).mean() for l in zip(*train_batch_losses)]
            train_losses.append(mean_train_losses)
            total_mean_train_batch_loss = mean_train_losses[0]

            mean_val_losses = [np.array(l).mean() for l in zip(*val_batch_losses)]
            val_losses.append(mean_val_losses)
            total_mean_val_batch_loss = mean_val_losses[0]

            if total_mean_val_batch_loss < best_val_loss:
                best_val_loss = total_mean_val_batch_loss
                best_model_dict = deepcopy(model.state_dict())
                run.summary["best_val_loss"] = best_val_loss    # save the validition loss so it can be seen in the dashboard

            print("Average Train Loss:", total_mean_train_batch_loss)
            print("Average Validation Loss:", total_mean_val_batch_loss)

            # because wandb.log() can only log scalars, we convert the list into a dict which will be then logged
            log_dict = {"epoch": epoch}
            for label, train_l, val_l in zip(labels, mean_train_losses, mean_val_losses):
                # create key-value pairs for each metric for the dict
                log_dict[f"train/{label}"] = train_l
                log_dict[f"val/{label}"] = val_l
            run.log(log_dict)


    torch.save(best_model_dict, f"best_models/TrafficSign_ObjDet_head_{best_val_loss:.4f}.pth")

    return train_losses, val_losses, best_val_loss


In [ ]:
learning_rate = 1e-3  # despite overshooting gradients quickly, especially on small objects, with warmup and lr decay later in training, a higher lr works well
epochs = 50
warmup_epochs = epochs * 0.2
weight_decay = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

warmup = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: min(1, (epoch + 1) / warmup_epochs))
lr_decay = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs)
lr_scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, lr_decay], [warmup_epochs])

# localization loss neads a smaller learning rate at the beginning, otherwise training will get unstable due to large gradients at the beginning and then small gradients due to small boxes with low IoU

In [ ]:
# train new head of the model (with frozen backbone) on the custom dataset

wandb_config = {
    "epochs": epochs,
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "no_object_weight": no_obj_w,
    "loc_weight": loc_w,
    "objectness_weight": obj_w,
    "wh_l2_factor": loc_l2_factor,
    "optimizer": type(optimizer).__name__,
    "lr_scheduler": lr_scheduler.state_dict()
}

train_losses, val_losses, best_val_loss = train_model(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    wandb_project_name="object-detection-resnet50",
    wandb_config=wandb_config, 
    epochs=epochs
)

print("Model reached best validation loss of:", best_val_loss)

In [ ]:
def plot_loss_curves(train_losses, val_losses):
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))

    labels = ["Objectness Loss", "Classification Loss", "Localization Loss"]
    colors = ["tab:blue", "tab:orange", "tab:green"]

    # Left: total loss
    axs[0].plot([epoch_losses[0] for epoch_losses in train_losses], label="Train Total", color="tab:blue")
    axs[0].plot([epoch_losses[0] for epoch_losses in val_losses], label="Val Total", color="tab:blue", linestyle="--")
    axs[0].set_title("Total Loss")
    axs[0].set_xlabel("Epochs")
    axs[0].set_ylabel("Loss")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # Right: component losses
    for idx, (lbl, color) in enumerate(zip(labels, colors), start=1):
        axs[1].plot([epoch_losses[idx] for epoch_losses in train_losses], label=f"Train {lbl}", color=color)
        axs[1].plot([epoch_losses[idx] for epoch_losses in val_losses], label=f"Val {lbl}", color=color, linestyle="--")

    axs[1].set_title("Component Losses")
    axs[1].set_xlabel("Epochs")
    axs[1].set_ylabel("Loss")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_loss_curves(train_losses, val_losses)